In [0]:
from pyspark.sql.functions import col, sum, count, round, avg

# Read silver tables
orders = spark.table("silver.olist.orders")
order_items = spark.table("silver.olist.order_items")
products = spark.table("silver.olist.products")

# Only delivered orders
delivered_orders = orders.filter(col("order_status") == "delivered") \
    .select("order_id")

# Join
df = order_items \
    .join(delivered_orders, on="order_id", how="inner") \
    .join(products.select("product_id", "product_category_name"), on="product_id", how="left") \
    .fillna({"product_category_name": "unknown"})

# Aggregate
gold_revenue = df.groupBy("product_category_name") \
    .agg(
        round(sum("price"), 2).alias("total_revenue"),
        round(avg("price"), 2).alias("avg_order_value"),
        count("order_id").alias("total_orders"),
        round(sum("freight_value"), 2).alias("total_freight")
    ) \
    .orderBy(col("total_revenue").desc())

gold_revenue.write.format("delta").mode("overwrite") \
    .option("path", "abfss://gold@project1storagek.dfs.core.windows.net/gold_revenue") \
    .saveAsTable("gold.olist.revenue_by_category")

print("revenue_by_category done:", gold_revenue.count())
gold_revenue.show(10)

revenue_by_category done: 72
+---------------------+-------------+---------------+------------+-------------+
|product_category_name|total_revenue|avg_order_value|total_orders|total_freight|
+---------------------+-------------+---------------+------------+-------------+
|        health_beauty|   1233131.72|         130.28|        9465|    178957.81|
|        watches_gifts|   1166176.98|         199.04|        5859|     98156.14|
|       bed_bath_table|   1023434.76|          93.44|       10953|     201774.5|
|       sports_leisure|    954852.55|         113.25|        8431|    163404.36|
| computers_accesso...|    888724.61|         116.26|        7644|    143999.16|
|      furniture_decor|    711927.69|          87.25|        8160|    168402.23|
|           housewares|    615628.69|           90.6|        6795|    142763.56|
|           cool_stuff|     610204.1|         164.12|        3718|     81476.79|
|                 auto|    578966.65|         139.85|        4140|      90488.1|

In [0]:
spark.sql("DROP TABLE IF EXISTS gold.olist.revenue_by_category")

DataFrame[]